# 01. Exploratory Data Analysis (EDA) - Credit Card Fraud Detection

## Project: Real-Time Credit Card Fraud Detection & Analytics System
**Objective**: Perform deep exploratory data analysis on the credit card transaction dataset, assess severe class imbalance, inspect transaction amounts, analyze temporal activity patterns, and evaluate PCA feature correlations.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.loader import load_transaction_data
df = load_transaction_data(source='auto')
print(f"Dataset Loaded. Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

### 1. Class Distribution Analysis
Understand the extreme imbalance between legitimate and fraudulent transactions.

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100

print(f"Legitimate Transactions (0): {class_counts[0]:,} ({100 - fraud_pct:.4f}%)")
print(f"Fraudulent Transactions (1): {class_counts[1]:,} ({fraud_pct:.4f}%)")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=['Legitimate (0)', 'Fraudulent (1)'], y=[class_counts[0], class_counts[1]], ax=ax[0], palette=['#2b5c8f', '#d9534f'])
ax[0].set_yscale('log')
ax[0].set_title('Transaction Count by Class (Log Scale)', fontsize=14)
ax[0].set_ylabel('Log Count')

ax[1].pie([class_counts[0], class_counts[1]], labels=['Legitimate', 'Fraud'], autopct='%1.2f%%', colors=['#2b5c8f', '#d9534f'], explode=[0, 0.2])
ax[1].set_title('Class Proportions', fontsize=14)
plt.tight_layout()
plt.show()

### 2. Transaction Amount Distribution
Examine transaction amount statistics across legitimate vs fraudulent cohorts.

In [ ]:
print("Legitimate Amount Summary:")
print(df[df['Class'] == 0]['Amount'].describe())
print("\nFraudulent Amount Summary:")
print(df[df['Class'] == 1]['Amount'].describe())

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df[df['Class'] == 0]['Amount'], bins=50, ax=ax[0], color='#2b5c8f', kde=True)
ax[0].set_title('Legitimate Transaction Amount Distribution')
ax[0].set_xlim(0, 1500)

sns.histplot(df[df['Class'] == 1]['Amount'], bins=50, ax=ax[1], color='#d9534f', kde=True)
ax[1].set_title('Fraudulent Transaction Amount Distribution')
ax[1].set_xlim(0, 1500)
plt.tight_layout()
plt.show()

### 3. Temporal Distribution (Time in Seconds to Hour of Day)
Convert elapsed seconds into hour of day (0 to 23) to assess whether fraud peaks during nighttime.

In [ ]:
df['hour'] = ((df['Time'] // 3600) % 24).astype(int)
hourly_summary = df.groupby(['hour', 'Class']).size().unstack(fill_value=0)
hourly_fraud_rate = (hourly_summary[1] / (hourly_summary[0] + hourly_summary[1]) * 100)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(hourly_summary.index, hourly_summary[0], alpha=0.4, color='#2b5c8f', label='Legit Transactions')
ax2.plot(hourly_fraud_rate.index, hourly_fraud_rate.values, color='#d9534f', marker='o', linewidth=2.5, label='Fraud Rate %')

ax1.set_xlabel('Hour of Day (0 - 23)', fontsize=12)
ax1.set_ylabel('Total Volume', color='#2b5c8f', fontsize=12)
ax2.set_ylabel('Fraud Rate (%)', color='#d9534f', fontsize=12)
ax1.set_title('Transaction Volume vs. Fraud Rate by Hour of Day', fontsize=14)
plt.show()

### 4. Correlation with Target (Class)
Identify which PCA components exhibit the strongest correlation with fraud.

In [ ]:
corrs = df.corr()['Class'].sort_values()
print("Top Negative Correlations with Class:")
print(corrs.head(6))
print("\nTop Positive Correlations with Class:")
print(corrs.tail(6))

plt.figure(figsize=(12, 6))
corrs[:-1].plot(kind='bar', color=np.where(corrs[:-1] > 0, '#d9534f', '#2b5c8f'))
plt.title('Feature Correlations with Fraud Target (Class)', fontsize=14)
plt.ylabel('Pearson Correlation Coefficient')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()